# Chạy bổ sung dữ liệu Gemma 2 9B

Notebook này tạo thêm học sinh mẫu bằng mô hình Gemma 2 9B, **nối tiếp** tệp đã có (`student_generated_gemma.csv`,
728 học sinh), tới 20 học sinh mỗi nhóm. Máy tự dừng sau khoảng 11 giờ sinh dữ liệu
(cộng khoảng 15-20 phút cài đặt và tải mô hình) và lưu kết quả, nên có thể bấm chạy rồi đi ngủ.

**Chuẩn bị (khoảng 10 phút):**

1. **Tài khoản Hugging Face:** đăng nhập https://huggingface.co, mở trang `google/gemma-2-9b-it` và bấm đồng ý điều khoản
   (nếu chưa). Tạo token đọc (Settings → Access Tokens → New token, quyền *Read*).
2. **Kaggle → Add-ons → Secrets:** thêm secret tên `HF_TOKEN`, giá trị là token ở bước 1, và bật cho notebook này.
3. **Tải tệp cũ lên làm dataset:** Kaggle → Datasets → New dataset → tải lên `student_generated_gemma.csv` (**giữ
   nguyên tên tệp**). Trong notebook: Add Input → chọn dataset vừa tạo.
4. **Cài đặt notebook (cột bên phải):** Accelerator = **GPU T4 x2**; Internet = **On**.

**Chạy:** bấm **Save Version** → **Save & Run All (Commit)** → Save. Không cần mở trình duyệt khi máy chạy.

**Sau khi chạy xong:** mở notebook → tab *Output* của phiên bản vừa chạy → tải `student_generated_gemma.csv` (và
`student_review_sample_gemma.csv` nếu có). Tệp mới đã gồm cả 728 học sinh cũ.

**Nếu log báo `No earlier gemma file is attached`:** notebook đã tự dừng (chưa tốn GPU). Gắn lại dataset (Add Input →
Your Work → Datasets) rồi chạy lại; tên tệp trong dataset có bị Kaggle đổi cũng không sao.

**Kiểm tra nhanh trong log:** dòng `gemma: N students from an earlier run kept, ... to generate` ở đầu, với N là số
học sinh trong tệp đã gắn (nếu ghi `0 students from an earlier run` là chưa gắn đúng dataset ở bước 3: dừng lại và gắn
lại). Sau đó:

- **Máy chạy nhanh (vLLM):** có dòng `vLLM: ... loaded in ... s`, rồi định kỳ `vLLM: .../... students, ... s`. Cuối cùng
  có dòng `vLLM wrote ... students in ... s (... s per student ...)`.
- **vLLM không chạy được mô hình này:** log ghi `vLLM stopped with exit code ...` kèm lỗi, rồi notebook tự chuyển sang
  cách chạy cũ (chậm hơn nhiều), với dòng `gemma: .../... students, ... parsed`. Không cần làm gì, nhưng hãy gửi
  lại đoạn log lỗi đó cho nhóm.

**Nên chạy thử trước (khoảng 15-30 phút):** đặt `TRIAL = True` trong ô cài đặt, chạy, xem log có dòng `vLLM wrote`
không, rồi đặt lại `TRIAL = False` và chạy thật.

# UniAdvisor: invented students for training the suggestion model ("Em chưa biết gì")

Open LLMs write invented grade-12 students who answer step 7's questionnaire (12 questions with fixed options, expected
exam scores, and the free text "Kể thêm về em"). Each prompt names one target **nhóm ngành**, so the label is known by
construction and the groups stay balanced (spec 3.5, "group first").

**Method (cited, verified 2026-10-08)**
- Prompting off-the-shelf LLMs with a target label to generate labelled training data: Sahu et al. 2022, *Data
  Augmentation for Intent Classification with Off-the-shelf Large Language Models*, 4th Workshop on NLP for
  Conversational AI (ACL 2022), pp. 47-57.
- Attributed prompts (each prompt samples how sure the student is, which questions they skip, how many options they
  tick, writing style, whether their best subjects point elsewhere) for diversity and less bias: Yu et al. 2023,
  *Large Language Model as Attributed Training Data Generator: A Tale of Diversity and Bias*, NeurIPS 2023 Datasets and
  Benchmarks.
- A short persona per prompt, so students of one group differ: Chan et al. 2024, *Scaling Synthetic Data Creation with
  1,000,000,000 Personas*, arXiv 2406.20094.
- Survey of generation, curation and evaluation: Long et al. 2024, *On LLMs-Driven Synthetic Data Generation,
  Curation, and Evaluation: A Survey*, Findings of ACL 2024, pp. 11065-11082.
- The questionnaire's interest options follow Holland's six types (Holland 1997, *Making Vocational Choices*, 3rd
  ed., PAR) and Prediger's work tasks (Prediger 1982, *Journal of Vocational Behavior* 21, 259-287), as used by the
  O*NET Interest Profiler (Lewis and Rivkin 1999; manual: Rounds, Hoff and Lewis 2021).
- Several LLM families; a held-out family for testing (spec 3.5). Training: Qwen 2.5 (Alibaba), Aya Expanse (Cohere),
  Gemma 2 (Google). Testing: SeaLLM-7B-v2 (DAMO Academy, built on Mistral 7B). SeaLLM's outputs are used only as test
  data, never to train or improve a language model (SeaLLMs License Agreement); notice: "SeaLLMs is licensed under
  the SeaLLMs LICENSE AGREEMENT, Copyright (c) Damo Academy."

**Before running (Kaggle)**: the same as the chat notebook. Accelerator *GPU T4 x2*, Internet *on*, the `HF_TOKEN`
secret with the Aya Expanse and Gemma 2 licences accepted. Run once with `TRIAL = True` (3 groups x 2 students, a few
minutes) and send the printed output or `student_generated_<family>_trial.csv` for a check before the long run. Choose the families in `RUN_FAMILIES` (one or two per
session keeps you inside Kaggle's time limit; each family is saved separately, and partial files are written every
few batches so a stopped session keeps its work).

**Output**: `/kaggle/working/student_generated_<family>.csv` (columns `student_id, target_group, family, model,
attributes, raw, answers`; `answers` is the parsed JSON, empty when the model's output was not valid JSON) and
`student_review_sample_<family>.csv` (3 students per group, for people to check). Download them into the repo's
`data/step7/generated/` (review samples into `data/step7/review/`).

In [ ]:
# ---- settings ----------------------------------------------------------------------------------------------
TRIAL = False                  # True: 3 groups x 2 students, to check the output and which engine ran
RUN_FAMILIES = ["gemma"]  # top-up run: Gemma 2 9B only
STUDENTS_PER_GROUP = {"qwen": 40, "aya": 40, "gemma": 20, "mistral": 15}   # 74 groups: about 3,000 per family
MAX_HOURS = 11.0               # top-up run: stop cleanly and save inside the GPU time left
REQUIRE_PREVIOUS = True        # top-up run: stop at once if the earlier gemma file is not attached
# Resume: attach an earlier student_generated_<family>.csv (as a Kaggle dataset); its students are kept and skipped.
MAX_NEW_TOKENS = 700           # answers are option codes (since 2026-10-09), shorter than the copied labels before
TEMPERATURE, TOP_P = 0.9, 0.95
SEED = 2027

MODELS = {
    # family: (model id, how the transformers engine loads it). fp16 on T4 (no bf16); Gemma 2 overflows in fp16.
    "qwen":    ("Qwen/Qwen2.5-7B-Instruct", "fp16"),
    "aya":     ("CohereLabs/aya-expanse-8b", "fp16"),
    "gemma":   ("google/gemma-2-9b-it", "4bit_fp32"),
    "mistral": ("SeaLLMs/SeaLLM-7B-v2", "fp16"),        # held-out test family (Mistral 7B base)
}

# Engines, tried in order. "vllm": continuous batching with paged attention (Kwon et al. 2023, SOSP), the model split
# over both T4s, up to 128 students at once, the shared questionnaire computed once (prefix caching): about 10x the
# first runs. It runs in its own process, so a model it cannot load falls through to "transformers" (the first runs'
# engine: slow, known to work for every family).
ENGINES = {"qwen": ["vllm", "transformers"], "aya": ["vllm", "transformers"], "gemma": ["vllm", "transformers"],
           "mistral": ["vllm", "transformers"]}
# max_num_batched_tokens: vLLM 0.28 doubled its default to 16384, more memory than a T4 has to spare
VLLM = {"tensor_parallel_size": 2, "dtype": "half", "max_model_len": 4096, "gpu_memory_utilization": 0.90,
        "max_num_seqs": 128, "max_num_batched_tokens": 8192, "enable_prefix_caching": True}
# Gemma 2 cannot run in fp16 (vLLM refuses it: overflow) and T4s have no bf16: 4-bit weights, fp32 compute. Since vLLM
# 0.28 bitsandbytes is a separate plugin (vllm-bnb-plugin, installed below; without it: "Unknown quantization method:
# bitsandbytes", gemma run of 2026-10-09). If vLLM cannot load it, the log says so and the transformers engine takes
# over.
VLLM_MODEL = {"gemma": {"dtype": "float32", "quantization": "bitsandbytes"}}
CHUNK = 256                    # vLLM: students per chunk; the results are saved after every chunk
BATCH_SIZE = 8                 # transformers engine: 16 runs out of memory on 2x T4 (runs of 2026-10-08)
SAVE_EVERY = 10                # transformers engine: batches between partial saves

In [ ]:
!pip install -q -U vllm vllm-bnb-plugin bitsandbytes accelerate
# Kaggle's preinstalled torchaudio can be built for another CUDA than the torch vllm brings, and transformers then fails
# on import; nothing here uses torchaudio
!pip uninstall -q -y torchaudio
import subprocess, sys
check = subprocess.run([sys.executable, "-c", "import torch, vllm, transformers; print('vllm', vllm.__version__, "
                        "'| torch', torch.__version__, '| transformers', transformers.__version__)"],
                       capture_output=True, text=True)
print(check.stdout or check.stderr[-3000:])
VLLM_OK = check.returncode == 0
if not VLLM_OK:
    print("vllm does not import (above): the transformers engine will be used, much slower")

In [ ]:
import gc, json, os, random, re, time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:   # not on Kaggle, or no secret: only ungated models will load
    print("No HF_TOKEN secret:", e)
print(torch.cuda.device_count(), "GPU(s)")

## Questionnaire, target groups, attributes and personas (from UniAdvisor's data)

In [ ]:
QUESTIONS = [{'id': 'q1',
  'text': 'Em thích học môn nào?',
  'kind': 'multi',
  'max': 3,
  'not_in': None,
  'codes': ['TO',
            'VA',
            'N1',
            'LI',
            'HO',
            'SI',
            'SU',
            'DI',
            'GDKTPL',
            'TI',
            'CNCN',
            'CNNN',
            'N3',
            'N4',
            'N6',
            'N7',
            'N5',
            'N2'],
  'options': ['Toán',
              'Ngữ văn',
              'Tiếng Anh',
              'Vật lí',
              'Hóa học',
              'Sinh học',
              'Lịch sử',
              'Địa lí',
              'Giáo dục kinh tế và pháp luật',
              'Tin học',
              'Công nghệ công nghiệp',
              'Công nghệ nông nghiệp',
              'Tiếng Pháp',
              'Tiếng Trung',
              'Tiếng Nhật',
              'Tiếng Hàn',
              'Tiếng Đức',
              'Tiếng Nga']},
 {'id': 'q2',
  'text': 'Môn nào em không thích?',
  'kind': 'multi',
  'max': 3,
  'not_in': 'q1',
  'codes': ['TO',
            'VA',
            'N1',
            'LI',
            'HO',
            'SI',
            'SU',
            'DI',
            'GDKTPL',
            'TI',
            'CNCN',
            'CNNN',
            'N3',
            'N4',
            'N6',
            'N7',
            'N5',
            'N2'],
  'options': ['Toán',
              'Ngữ văn',
              'Tiếng Anh',
              'Vật lí',
              'Hóa học',
              'Sinh học',
              'Lịch sử',
              'Địa lí',
              'Giáo dục kinh tế và pháp luật',
              'Tin học',
              'Công nghệ công nghiệp',
              'Công nghệ nông nghiệp',
              'Tiếng Pháp',
              'Tiếng Trung',
              'Tiếng Nhật',
              'Tiếng Hàn',
              'Tiếng Đức',
              'Tiếng Nga']},
 {'id': 'q3',
  'text': 'Em thích làm việc với điều gì?',
  'kind': 'multi',
  'max': 2,
  'not_in': None,
  'codes': ['people', 'machines', 'data', 'ideas'],
  'options': ['Con người', 'Máy móc', 'Số liệu', 'Ý tưởng']},
 {'id': 'q4',
  'text': 'Em thích làm những việc nào?',
  'kind': 'multi',
  'max': None,
  'not_in': None,
  'codes': ['repair',
            'electronics',
            'code',
            'experiment',
            'hard_problem',
            'how_machines_work',
            'plants',
            'animals',
            'cooking',
            'sport',
            'drawing',
            'photo',
            'video',
            'stories',
            'instrument'],
  'options': ['Sửa chữa đồ đạc',
              'Lắp ráp đồ điện tử',
              'Viết code',
              'Làm thí nghiệm',
              'Giải bài toán khó',
              'Tìm hiểu máy móc hoạt động ra sao',
              'Trồng cây',
              'Chăm sóc con vật',
              'Nấu ăn',
              'Chơi thể thao',
              'Vẽ',
              'Chụp ảnh',
              'Làm video',
              'Viết truyện',
              'Chơi nhạc cụ']},
 {'id': 'q5',
  'text': 'Còn những việc này?',
  'kind': 'multi',
  'max': None,
  'not_in': None,
  'codes': ['tutoring',
            'listening',
            'caring_sick',
            'volunteering',
            'events',
            'presenting',
            'debating',
            'selling',
            'budgeting',
            'filing',
            'novels',
            'news'],
  'options': ['Giảng bài cho bạn',
              'Nghe bạn bè tâm sự',
              'Chăm sóc người ốm',
              'Làm tình nguyện',
              'Tổ chức sự kiện',
              'Thuyết trình',
              'Tranh luận',
              'Bán hàng',
              'Quản lý chi tiêu',
              'Sắp xếp tài liệu',
              'Đọc tiểu thuyết',
              'Theo dõi thời sự']},
 {'id': 'q6',
  'text': 'Em thích tìm hiểu về chủ đề nào?',
  'kind': 'multi',
  'max': 4,
  'not_in': None,
  'codes': ['computers',
            'electronics_topic',
            'mechanics',
            'construction',
            'health',
            'psychology',
            'society',
            'economics',
            'marketing',
            'business',
            'law',
            'journalism',
            'literature',
            'languages',
            'history',
            'geography',
            'arts',
            'agriculture',
            'cuisine',
            'environment',
            'transport',
            'education',
            'philosophy'],
  'options': ['Máy tính',
              'Điện tử',
              'Cơ khí',
              'Xây dựng',
              'Sức khỏe',
              'Tâm lý',
              'Các vấn đề xã hội',
              'Kinh tế',
              'Marketing',
              'Quản lý doanh nghiệp',
              'Pháp luật',
              'Báo chí',
              'Văn chương',
              'Ngoại ngữ',
              'Lịch sử',
              'Địa lý',
              'Nghệ thuật',
              'Nông nghiệp',
              'Ẩm thực',
              'Môi trường',
              'Giao thông vận tải',
              'Giáo dục',
              'Triết học']},
 {'id': 'q7',
  'text': 'Có chủ đề nào em chắc chắn không muốn học?',
  'kind': 'multi',
  'max': 3,
  'not_in': 'q6',
  'codes': ['computers',
            'electronics_topic',
            'mechanics',
            'construction',
            'health',
            'psychology',
            'society',
            'economics',
            'marketing',
            'business',
            'law',
            'journalism',
            'literature',
            'languages',
            'history',
            'geography',
            'arts',
            'agriculture',
            'cuisine',
            'environment',
            'transport',
            'education',
            'philosophy'],
  'options': ['Máy tính',
              'Điện tử',
              'Cơ khí',
              'Xây dựng',
              'Sức khỏe',
              'Tâm lý',
              'Các vấn đề xã hội',
              'Kinh tế',
              'Marketing',
              'Quản lý doanh nghiệp',
              'Pháp luật',
              'Báo chí',
              'Văn chương',
              'Ngoại ngữ',
              'Lịch sử',
              'Địa lý',
              'Nghệ thuật',
              'Nông nghiệp',
              'Ẩm thực',
              'Môi trường',
              'Giao thông vận tải',
              'Giáo dục',
              'Triết học']},
 {'id': 'q8',
  'text': 'Sau này em muốn làm việc chủ yếu ở đâu?',
  'kind': 'multi',
  'max': 3,
  'not_in': None,
  'codes': ['office',
            'workshop',
            'outdoors',
            'vehicle',
            'hospital',
            'school',
            'lab',
            'site',
            'farm',
            'hotel',
            'shop',
            'home'],
  'options': ['Văn phòng',
              'Nhà xưởng',
              'Ngoài trời',
              'Trên phương tiện giao thông',
              'Bệnh viện',
              'Trường học',
              'Phòng thí nghiệm',
              'Công trường',
              'Trang trại',
              'Khách sạn',
              'Cửa hàng',
              'Tại nhà']},
 {'id': 'q9',
  'text': 'Em thích làm việc thế nào?',
  'kind': 'single',
  'max': None,
  'not_in': None,
  'codes': ['alone', 'team', 'new_people'],
  'options': ['Một mình', 'Cùng một nhóm', 'Gặp gỡ nhiều người mới']},
 {'id': 'q10',
  'text': 'Em thích công việc như thế nào hơn?',
  'kind': 'single',
  'max': None,
  'not_in': None,
  'codes': ['procedure', 'own_way'],
  'options': ['Làm theo quy trình rõ ràng', 'Tự nghĩ ra cách làm']},
 {'id': 'q11',
  'text': 'Điều gì quan trọng nhất với em trong công việc sau này?',
  'kind': 'single',
  'max': None,
  'not_in': None,
  'codes': ['helping', 'making', 'understanding', 'expressing', 'leading', 'accuracy'],
  'options': ['Giúp đỡ người khác',
              'Làm ra sản phẩm cầm nắm được',
              'Hiểu vì sao mọi thứ xảy ra',
              'Thể hiện ý tưởng của mình',
              'Dẫn dắt người khác',
              'Làm việc chính xác']},
 {'id': 'q12',
  'text': 'Em không muốn công việc có điều nào?',
  'kind': 'multi',
  'max': None,
  'not_in': None,
  'codes': ['blood',
            'sick_people',
            'night_shift',
            'away_from_home',
            'sitting',
            'standing',
            'public_speaking',
            'outdoor_work',
            'heights',
            'chemicals',
            'noise',
            'time_pressure',
            'angry_people',
            'overtime',
            'sales_targets'],
  'options': ['Thấy máu',
              'Tiếp xúc với người bệnh',
              'Làm ca đêm',
              'Thường xuyên đi xa nhà',
              'Ngồi một chỗ cả ngày',
              'Đứng cả ngày',
              'Nói trước đông người',
              'Làm việc ngoài trời',
              'Làm việc trên cao',
              'Tiếp xúc hóa chất',
              'Tiếng ồn lớn',
              'Áp lực thời gian',
              'Đối mặt với người nóng giận',
              'Làm thêm giờ thường xuyên',
              'Chạy doanh số']}]

GROUPS = [{'code': '71401',
  'name': 'Quản lý giáo dục, Công nghệ giáo dục',
  'nganh': ['Quản lý giáo dục', 'Công nghệ giáo dục', 'Giáo dục học']},
 {'code': '71402',
  'name': 'Sư phạm (đào tạo giáo viên)',
  'nganh': ['Giáo dục Tiểu học',
            'Sư phạm Toán học',
            'Sư phạm Ngữ văn',
            'Sư phạm Tiếng Anh',
            'Sư phạm Vật lý',
            'Sư phạm Lịch sử',
            'Sư phạm công nghệ',
            'Sư phạm Khoa học tự nhiên']},
 {'code': '72101',
  'name': 'Mỹ thuật',
  'nganh': ['Lý luận, lịch sử và phê bình mỹ thuật', 'Hội họa', 'Đồ họa', 'Điêu khắc']},
 {'code': '72102',
  'name': 'Thanh nhạc, Nghệ thuật trình diễn',
  'nganh': ['Thanh nhạc',
            'Âm nhạc học',
            'Sáng tác âm nhạc',
            'Chỉ huy âm nhạc',
            'Biểu diễn nhạc cụ phương tây',
            'Piano',
            'Biểu diễn nhạc cụ truyền thống',
            'Nhạc Jazz']},
 {'code': '72103',
  'name': 'Điện ảnh - truyền hình, Thiết kế âm thanh - ánh sáng',
  'nganh': ['Công nghệ điện ảnh, truyền hình', 'Thiết kế âm thanh, ánh sáng']},
 {'code': '72104',
  'name': 'Thiết kế đồ họa, Thiết kế thời trang, Nghệ thuật số',
  'nganh': ['Thiết kế đồ họa', 'Thiết kế thời trang', 'Digital Art (Nghệ thuật số)', 'Thiết kế công nghiệp']},
 {'code': '72201',
  'name': 'Văn hóa Việt Nam, Hán Nôm',
  'nganh': ['Tiếng Việt và văn hóa Việt Nam',
            'Hán Nôm',
            'Văn hóa các dân tộc thiểu số Việt Nam',
            'Sáng tác văn học']},
 {'code': '72202',
  'name': 'Ngôn ngữ Anh, Trung, Hàn, Nhật... (ngoại ngữ)',
  'nganh': ['Ngôn ngữ Anh',
            'Ngôn ngữ Trung Quốc',
            'Ngôn ngữ Hàn Quốc',
            'Ngôn ngữ Nhật',
            'Ngôn ngữ Nga',
            'Ngôn ngữ Pháp',
            'Ngôn ngữ Đức',
            'Ngôn ngữ Italia']},
 {'code': '72290',
  'name': 'Văn hóa học, Quản lý văn hóa, Văn học, Lịch sử, Triết học',
  'nganh': ['Văn hóa học',
            'Quản lý văn hóa',
            'Văn học',
            'Lịch sử',
            'Triết học',
            'Ngôn ngữ học',
            'Di sản học',
            'Tôn giáo học']},
 {'code': '73101',
  'name': 'Kinh tế, Kinh tế quốc tế, Kinh tế đầu tư',
  'nganh': ['Kinh tế',
            'Kinh tế quốc tế',
            'Kinh tế số',
            'Toán kinh tế',
            'Kinh tế đầu tư',
            'Kinh tế chính trị',
            'Kinh tế phát triển',
            'Thống kê kinh tế']},
 {'code': '73102',
  'name': 'Quan hệ quốc tế, Quản lý nhà nước, Chính trị học',
  'nganh': ['Quản lý nhà nước', 'Quan hệ quốc tế', 'Xây dựng Đảng và chính quyền nhà nước', 'Chính trị học']},
 {'code': '73103', 'name': 'Xã hội học và Nhân học', 'nganh': ['Xã hội học', 'Nhân học']},
 {'code': '73104', 'name': 'Tâm lý học', 'nganh': ['Tâm lý học', 'Tâm lý học giáo dục']},
 {'code': '73105', 'name': 'Địa lý học', 'nganh': ['Địa lý học']},
 {'code': '73106',
  'name': 'Việt Nam học, Quốc tế học, Đông phương học',
  'nganh': ['Việt Nam học',
            'Đông phương học',
            'Quốc tế học',
            'Nhật Bản học',
            'Hàn Quốc học',
            'Trung Quốc học',
            'Đông Nam Á học']},
 {'code': '73201',
  'name': 'Báo chí, Truyền thông đa phương tiện, Quan hệ công chúng (PR)',
  'nganh': ['Truyền thông đa phương tiện',
            'Quan hệ công chúng',
            'Báo chí',
            'Công nghệ truyền thông',
            'Truyền thông đại chúng',
            'Truyền thông quốc tế']},
 {'code': '73202', 'name': 'Thông tin - Thư viện', 'nganh': ['Thông tin - Thư viện', 'Quản lý thông tin']},
 {'code': '73203', 'name': 'Văn thư - Lưu trữ - Bảo tàng', 'nganh': ['Lưu trữ học', 'Bảo tàng học']},
 {'code': '73204', 'name': 'Xuất bản - Phát hành', 'nganh': ['Kinh doanh xuất bản phẩm']},
 {'code': '73290', 'name': 'Công nghệ đa phương tiện', 'nganh': ['Công nghệ đa phương tiện']},
 {'code': '73401',
  'name': 'Quản trị kinh doanh, Marketing, Kinh doanh quốc tế, Thương mại điện tử',
  'nganh': ['Quản trị kinh doanh',
            'Marketing',
            'Kinh doanh quốc tế',
            'Thương mại điện tử',
            'Kinh doanh thương mại',
            'Bất động sản',
            'Digital Marketing',
            'Quản lý']},
 {'code': '73402',
  'name': 'Tài chính - Ngân hàng, Công nghệ tài chính (Fintech), Bảo hiểm',
  'nganh': ['Tài chính - Ngân hàng', 'Công nghệ tài chính', 'Bảo hiểm']},
 {'code': '73403', 'name': 'Kế toán - Kiểm toán', 'nganh': ['Kế toán', 'Kiểm toán']},
 {'code': '73404',
  'name': 'Hệ thống thông tin quản lý, Quản trị nhân lực, Quản trị sự kiện',
  'nganh': ['Quản trị nhân lực',
            'Hệ thống thông tin quản lý',
            'Quản trị văn phòng',
            'Quản lý công',
            'Khoa học quản lý',
            'Quản trị sự kiện',
            'Quan hệ lao động',
            'Quản lý dự án']},
 {'code': '73801',
  'name': 'Luật, Luật kinh tế, Luật quốc tế',
  'nganh': ['Luật',
            'Luật kinh tế',
            'Luật thương mại quốc tế',
            'Luật quốc tế',
            'Luật dân sự và tố tụng dân sự',
            'Luật hình sự và tố tụng hình sự']},
 {'code': '74201', 'name': 'Sinh học', 'nganh': ['Sinh học']},
 {'code': '74202',
  'name': 'Công nghệ sinh học',
  'nganh': ['Công nghệ sinh học', 'Sinh học ứng dụng', 'Công nghệ thẩm mỹ', 'Kỹ thuật sinh học']},
 {'code': '74290', 'name': 'Sinh dược học', 'nganh': ['Sinh dược học']},
 {'code': '74401',
  'name': 'Hóa học, Vật lý học, Khoa học vật liệu',
  'nganh': ['Hóa học', 'Khoa học vật liệu', 'Vật lý học', 'Thiên văn học']},
 {'code': '74402',
  'name': 'Địa chất học, Hải dương học, Khí tượng - thủy văn',
  'nganh': ['Địa chất học',
            'Thủy văn học',
            'Khí tượng và khí hậu học',
            'Hải dương học',
            'Địa lý tự nhiên',
            'Quản lý và phân tích dữ liệu khoa học',
            'Khoa học thông tin địa không gian']},
 {'code': '74403', 'name': 'Khoa học môi trường', 'nganh': ['Khoa học môi trường']},
 {'code': '74601',
  'name': 'Khoa học dữ liệu, Toán ứng dụng, Toán tin',
  'nganh': ['Khoa học dữ liệu', 'Toán ứng dụng', 'Toán tin', 'Toán học', 'Khoa học tính toán']},
 {'code': '74602', 'name': 'Thống kê', 'nganh': ['Thống kê']},
 {'code': '74801',
  'name': 'Khoa học máy tính, Trí tuệ nhân tạo (AI), Kỹ thuật phần mềm',
  'nganh': ['Khoa học máy tính',
            'Trí tuệ nhân tạo',
            'Kỹ thuật phần mềm',
            'Hệ thống thông tin',
            'Mạng máy tính và truyền thông dữ liệu',
            'Kỹ thuật máy tính',
            'Công nghệ kỹ thuật máy tính',
            'Khoa học máy tính và thông tin']},
 {'code': '74802',
  'name': 'Công nghệ thông tin, An toàn thông tin, An ninh mạng',
  'nganh': ['Công nghệ thông tin',
            'An toàn thông tin',
            'An ninh mạng',
            'Kỹ thuật dữ liệu',
            'Trí tuệ nhân tạo và khoa học dữ liệu',
            'Địa tin học']},
 {'code': '75101',
  'name': 'Công nghệ kỹ thuật xây dựng',
  'nganh': ['Công nghệ kỹ thuật công trình xây dựng',
            'Công nghệ kỹ thuật giao thông',
            'Công nghệ kỹ thuật xây dựng',
            'Công nghệ kỹ thuật vật liệu xây dựng',
            'Hệ thống kỹ thuật công trình xây dựng']},
 {'code': '75102',
  'name': 'Công nghệ kỹ thuật cơ khí, cơ điện tử, ô tô',
  'nganh': ['Công nghệ kỹ thuật ô tô',
            'Công nghệ kỹ thuật cơ khí',
            'Công nghệ kỹ thuật cơ điện tử',
            'Công nghệ chế tạo máy',
            'Công nghệ kỹ thuật nhiệt',
            'Năng lượng tái tạo',
            'Robot và trí tuệ nhân tạo',
            'Bảo dưỡng công nghiệp']},
 {'code': '75103',
  'name': 'Công nghệ kỹ thuật điện, điện tử, tự động hóa',
  'nganh': ['Công nghệ kỹ thuật điện, điện tử',
            'Công nghệ kỹ thuật điện tử - viễn thông',
            'Công nghệ kỹ thuật điều khiển và tự động hóa']},
 {'code': '75104',
  'name': 'Công nghệ hóa học, môi trường, vật liệu',
  'nganh': ['Công nghệ kỹ thuật môi trường',
            'Công nghệ kỹ thuật hóa học',
            'Công nghệ vật liệu',
            'Công nghệ kỹ thuật hạt nhân']},
 {'code': '75106',
  'name': 'Logistics và Quản lý chuỗi cung ứng, Quản lý công nghiệp',
  'nganh': ['Logistics và Quản lý chuỗi cung ứng', 'Quản lý công nghiệp', 'Kinh tế công nghiệp']},
 {'code': '75108', 'name': 'Công nghệ kỹ thuật in', 'nganh': ['Công nghệ kỹ thuật in']},
 {'code': '75190',
  'name': 'Công nghệ hàng không vũ trụ, Công nghệ nông nghiệp',
  'nganh': ['Công nghệ hàng không vũ trụ', 'Công nghệ nông nghiệp']},
 {'code': '75201',
  'name': 'Kỹ thuật cơ khí, cơ điện tử, ô tô',
  'nganh': ['Kỹ thuật cơ điện tử',
            'Kỹ thuật cơ khí',
            'Kỹ thuật ô tô',
            'Kỹ thuật hàng không',
            'Kỹ thuật nhiệt',
            'Kỹ thuật cơ khí động lực',
            'Kỹ thuật hệ thống công nghiệp',
            'Kỹ thuật Robot']},
 {'code': '75202',
  'name': 'Kỹ thuật điện, điện tử - viễn thông, tự động hóa',
  'nganh': ['Kỹ thuật điện tử - viễn thông',
            'Kỹ thuật điều khiển và tự động hóa',
            'Kỹ thuật điện',
            'Kỹ thuật y sinh',
            'Kỹ thuật điện tử và tin học',
            'Kỹ thuật Robot và Điều khiển thông minh']},
 {'code': '75203',
  'name': 'Kỹ thuật môi trường, hóa học, vật liệu',
  'nganh': ['Kỹ thuật hoá học', 'Kỹ thuật môi trường', 'Kỹ thuật vật liệu', 'Kỹ thuật Bán dẫn', 'Kỹ thuật dệt']},
 {'code': '75204',
  'name': 'Vật lý kỹ thuật',
  'nganh': ['Vật lý kỹ thuật', 'Kỹ thuật hạt nhân', 'Kỹ thuật năng lượng']},
 {'code': '75205',
  'name': 'Kỹ thuật địa chất, trắc địa - bản đồ',
  'nganh': ['Kỹ thuật địa chất', 'Kỹ thuật trắc địa - bản đồ', 'Kỹ thuật địa vật lý', 'Đá quý Đá mỹ nghệ']},
 {'code': '75206',
  'name': 'Kỹ thuật mỏ, dầu khí',
  'nganh': ['Kỹ thuật dầu khí',
            'Kỹ thuật mỏ',
            'Kỹ thuật khí thiên nhiên',
            'Công nghệ số trong thăm dò và khai thác tài nguyên thiên nhiên',
            'Kỹ thuật tuyển khoáng']},
 {'code': '75401',
  'name': 'Công nghệ thực phẩm',
  'nganh': ['Công nghệ thực phẩm',
            'Đảm bảo chất lượng và an toàn thực phẩm',
            'Kỹ thuật thực phẩm',
            'Công nghệ chế biến thủy sản',
            'Khoa học và công nghệ thực phẩm']},
 {'code': '75402',
  'name': 'Công nghệ dệt, may',
  'nganh': ['Công nghệ dệt, may', 'Công nghệ vật liệu dệt, may', 'Công nghệ may', 'Công nghệ sợi, dệt']},
 {'code': '75490',
  'name': 'Công nghệ chế biến lâm sản, Kỹ nghệ gỗ và nội thất',
  'nganh': ['Công nghệ chế biến lâm sản', 'Kỹ nghệ gỗ và nội thất']},
 {'code': '75801',
  'name': 'Kiến trúc, Quy hoạch, Thiết kế nội thất',
  'nganh': ['Kiến trúc',
            'Quản lý đô thị và công trình',
            'Thiết kế nội thất',
            'Quản lý phát triển đô thị và bất động sản',
            'Kiến trúc đô thị',
            'Quy hoạch vùng và đô thị',
            'Kiến trúc cảnh quan',
            'Đô thị học']},
 {'code': '75802',
  'name': 'Kỹ thuật xây dựng, Công trình giao thông',
  'nganh': ['Kỹ thuật xây dựng',
            'Kỹ thuật xây dựng công trình giao thông',
            'Kỹ thuật cơ sở hạ tầng',
            'Kỹ thuật xây dựng công trình thủy',
            'Kỹ thuật cấp thoát nước',
            'Kỹ thuật tài nguyên nước',
            'Địa kỹ thuật xây dựng',
            'Xây dựng công trình ngầm thành phố và Hệ thống tàu điện ngầm']},
 {'code': '75803', 'name': 'Quản lý xây dựng, Kinh tế xây dựng', 'nganh': ['Quản lý xây dựng', 'Kinh tế xây dựng']},
 {'code': '76201',
  'name': 'Nông nghiệp',
  'nganh': ['Chăn nuôi',
            'Kinh doanh nông nghiệp',
            'Khoa học cây trồng',
            'Chăn nuôi thú y - Thuỷ sản',
            'Nông học',
            'Bảo vệ thực vật',
            'Kinh tế nông nghiệp',
            'Phát triển nông thôn']},
 {'code': '76202',
  'name': 'Lâm nghiệp',
  'nganh': ['Quản lý tài nguyên rừng', 'Lâm học', 'Lâm nghiệp đô thị', 'Lâm sinh']},
 {'code': '76203', 'name': 'Nuôi trồng thủy sản', 'nganh': ['Nuôi trồng thủy sản']},
 {'code': '76401', 'name': 'Thú y', 'nganh': ['Thú y']},
 {'code': '77201',
  'name': 'Y khoa (bác sĩ), Y học cổ truyền',
  'nganh': ['Y khoa', 'Y học cổ truyền', 'Y học dự phòng']},
 {'code': '77202', 'name': 'Dược học, Hóa dược', 'nganh': ['Dược học', 'Hóa dược']},
 {'code': '77203', 'name': 'Điều dưỡng - hộ sinh', 'nganh': ['Điều dưỡng', 'Hộ sinh']},
 {'code': '77204', 'name': 'Dinh dưỡng', 'nganh': ['Dinh dưỡng']},
 {'code': '77205', 'name': 'Nha khoa (Răng - Hàm - Mặt)', 'nganh': ['Răng - Hàm - Mặt', 'Kỹ thuật phục hình răng']},
 {'code': '77206',
  'name': 'Xét nghiệm, Hình ảnh y học, Phục hồi chức năng',
  'nganh': ['Kỹ thuật xét nghiệm y học', 'Kỹ thuật Phục hồi chức năng', 'Kỹ thuật hình ảnh y học']},
 {'code': '77207', 'name': 'Y tế công cộng', 'nganh': ['Y tế công cộng']},
 {'code': '77208', 'name': 'Quản lý Y tế', 'nganh': ['Tổ chức và Quản lý y tế', 'Quản lý bệnh viện']},
 {'code': '77601',
  'name': 'Công tác xã hội',
  'nganh': ['Công tác xã hội', 'Công tác thanh thiếu niên', 'Hỗ trợ giáo dục người khuyết tật']},
 {'code': '78101',
  'name': 'Du lịch, Quản trị dịch vụ du lịch và lữ hành',
  'nganh': ['Quản trị dịch vụ du lịch và lữ hành', 'Du lịch', 'Du lịch địa chất']},
 {'code': '78102',
  'name': 'Quản trị khách sạn, nhà hàng',
  'nganh': ['Quản trị khách sạn', 'Quản trị nhà hàng và dịch vụ ăn uống']},
 {'code': '78103', 'name': 'Quản lý thể dục thể thao', 'nganh': ['Quản lý thể dục thể thao']},
 {'code': '78401',
  'name': 'Khai thác và kinh tế vận tải',
  'nganh': ['Khai thác vận tải', 'Kinh tế vận tải', 'Quản lý hoạt động bay', 'Quản lý và vận hành hạ tầng']},
 {'code': '78501',
  'name': 'Quản lý tài nguyên và môi trường, Quản lý đất đai',
  'nganh': ['Quản lý tài nguyên và môi trường',
            'Quản lý đất đai',
            'Kinh tế tài nguyên thiên nhiên',
            'Quản lý tài nguyên khoáng sản']},
 {'code': '78502', 'name': 'An toàn, vệ sinh lao động', 'nganh': ['Bảo hộ lao động', 'An toàn, Vệ sinh lao động']},
 {'code': '78590', 'name': 'Môi trường, sức khỏe và an toàn', 'nganh': ['Môi trường, sức khỏe và an toàn']}]

SUBJECTS = ['Toán',
 'Ngữ văn',
 'Tiếng Anh',
 'Vật lí',
 'Hóa học',
 'Sinh học',
 'Lịch sử',
 'Địa lí',
 'Giáo dục kinh tế và pháp luật',
 'Tin học',
 'Công nghệ công nghiệp',
 'Công nghệ nông nghiệp',
 'Tiếng Pháp',
 'Tiếng Trung',
 'Tiếng Nhật',
 'Tiếng Hàn',
 'Tiếng Đức',
 'Tiếng Nga']

ATTRIBUTES = {'clarity': ['rất rõ mình thích gì', 'khá rõ mình thích gì', 'còn phân vân, thích cả vài hướng khác nhau'],
 'subjects': ['các môn em học tốt nhất hợp với nhóm ngành này',
              'các môn em học tốt nhất hợp với nhóm ngành này',
              'các môn em học tốt nhất lại không liên quan nhiều đến nhóm ngành này'],
 'skips': ['trả lời đủ mọi câu', 'trả lời đủ mọi câu', 'bỏ qua 1-2 câu em thấy khó', 'bỏ qua 3-4 câu'],
 'picks': ['chọn ít, chỉ những gì thật sự đúng', 'chọn vừa phải', 'chọn khá nhiều'],
 'level': ['học lực trung bình', 'học lực khá', 'học lực khá', 'học lực giỏi'],
 'free_text': ['để trống', 'một câu ngắn', 'hai ba câu', 'một đoạn dài, kể cả hoàn cảnh'],
 'free_text_style': ['có dấu đầy đủ', 'có dấu đầy đủ', 'không dấu', 'viết tắt kiểu teencode (k, ko, dc, e, j)'],
 'mentions_job': ['không nhắc tên nghề nào', 'không nhắc tên nghề nào', 'có nhắc một nghề em mơ ước']}

PERSONAS = ['học sinh một trường THPT ở thị trấn nhỏ',
 'học sinh trường chuyên ở thành phố lớn',
 'học sinh hay tham gia câu lạc bộ ở trường',
 'học sinh ít nói, thích ở nhà',
 'học sinh hay phụ giúp việc của gia đình',
 'học sinh thích đọc sách và tìm hiểu trên mạng',
 'học sinh từng đi làm thêm vào dịp hè',
 'học sinh có bố mẹ làm nông',
 'học sinh có bố mẹ làm công nhân',
 'học sinh có bố mẹ làm nhân viên văn phòng',
 'học sinh có bố mẹ buôn bán nhỏ',
 'học sinh học ở miền núi',
 'học sinh hay làm lớp trưởng hoặc cán bộ lớp',
 'học sinh mê một môn thể thao',
 'học sinh thích công nghệ mới']

## Generation

In [ ]:
import glob, json, random, re, subprocess, sys, time
random.seed(SEED)

# How many options a student ticks per multi-choice question, by the "picks" attribute (a range; capped by the
# question's max). Disliked subjects / topics (q2, q7) and "không muốn" (q12) are ticked less.
PICK_RANGE = {"chọn ít, chỉ những gì thật sự đúng": (1, 2), "chọn vừa phải": (2, 3), "chọn khá nhiều": (3, 5)}
FEWER = {"q2", "q7", "q12"}
SKIP_RANGE = {"trả lời đủ mọi câu": (0, 0), "bỏ qua 1-2 câu em thấy khó": (1, 2), "bỏ qua 3-4 câu": (3, 4)}

def plan_answers(attrs, rng):
    # The attributes as concrete instructions per question (the trial run showed that "chọn ít" alone is ignored).
    skip = set(rng.sample([q["id"] for q in QUESTIONS if q["id"] != "q1"], rng.randint(*SKIP_RANGE[attrs["skips"]])))
    counts = {}
    for q in QUESTIONS:
        if q["id"] in skip:
            counts[q["id"]] = None
        elif q["kind"] == "single":
            counts[q["id"]] = 1
        else:
            lo, hi = PICK_RANGE[attrs["picks"]]
            if q["id"] in FEWER:
                lo, hi = max(0, lo - 1), max(1, hi - 2)
            counts[q["id"]] = min(rng.randint(lo, hi), q.get("max") or 99, len(q["options"]))
    return counts

# A question whose options are those of an earlier one (q2 = q1, q7 = q6) points to it instead of repeating them.
SAME_AS = {}
for i, q in enumerate(QUESTIONS):
    for p in QUESTIONS[:i]:
        if q["codes"] == p["codes"] and p["id"] not in SAME_AS:
            SAME_AS[q["id"]] = p["id"]
            break

def questionnaire_text():
    lines = []
    for q in QUESTIONS:
        kind = "chọn một mã" if q["kind"] == "single" else "chọn nhiều mã"
        rule = f", không trùng với {q['not_in']}" if q.get("not_in") else ""
        opts = (f"dùng các mã của {SAME_AS[q['id']]}" if q["id"] in SAME_AS else
                " | ".join(f"{c}={label}" for c, label in zip(q["codes"], q["options"])))
        lines.append(f"{q['id']}. {q['text']} ({kind}{rule}): {opts}")
    return "\n".join(lines)

# The part every prompt shares, first, so vLLM computes it once (prefix caching); the student comes after it. The
# writer answers with option codes, not labels: shorter outputs and fewer copying mistakes (v1's generator,
# 2026-10-09). The example shows the shape only (a trial writer once copied an example score).
PREFIX = (
    "Bạn giúp tạo dữ liệu huấn luyện cho một ứng dụng tư vấn chọn ngành đại học ở Việt Nam. Ở cuối có mô tả MỘT học "
    "sinh lớp 12: hãy tưởng tượng học sinh đó và điền bảng câu hỏi như em ấy tự điền.\n\n"
    "BẢNG CÂU HỎI. Mỗi lựa chọn viết dạng MÃ=nội dung. Trả lời bằng MÃ, chép đúng mã, chỉ dùng mã của chính câu đó.\n"
    f"{questionnaire_text()}\n"
    "diem: điểm thi dự kiến, một đối tượng {MÃ môn: điểm}, mã môn lấy từ q1; mỗi điểm từ 0 đến 10, bước 0,25, hợp với "
    "học lực và các môn em học tốt; điểm như học sinh thật, mỗi môn một điểm khác nhau.\n"
    "ke_them: phần 'Kể thêm về em', học sinh tự viết bằng tiếng Việt, xưng 'em' hoặc 'mình'. Không nhắc tên nhóm "
    "ngành hay tên ngành. Không chép lại các dòng mô tả học sinh (trường, học lực, mức độ phân vân): viết điều em "
    "thích, điều em không muốn.\n\n"
    "QUY TẮC\n"
    "- Chọn ĐÚNG số mã ghi cho từng câu ở phần mô tả học sinh. Câu BỎ QUA: null. Câu 'không chọn gì': [].\n"
    "- Câu chọn nhiều: một mảng JSON các mã. Câu chọn một: một chuỗi mã.\n"
    "- Học sinh thật không hoàn hảo: không chọn mọi thứ liên quan đến ngành, có thể thích cả vài điều không liên quan.\n"
    '- Chỉ trả về MỘT đối tượng JSON với các khóa "q1" đến "q12", "diem" và "ke_them". Không giải thích.\n'
    'Dạng (không phải nội dung): {"q1": ["<mã>", "<mã>"], "q2": null, ..., "q9": "<mã>", ..., '
    '"diem": {"<mã môn>": <điểm>}, "ke_them": "<chuỗi>"}\n\n'
)

def plan_text(counts):
    parts = []
    for q in QUESTIONS:
        k = counts[q["id"]]
        how = ("BỎ QUA (null)" if k is None else "chọn 1 mã" if q["kind"] == "single" else
               "không chọn gì ([])" if k == 0 else f"chọn đúng {k} mã")
        parts.append(f"{q['id']}: {how}")
    return "; ".join(parts)

def build_prompt(group, rng):
    attrs = {k: rng.choice(v) for k, v in ATTRIBUTES.items()}
    attrs["persona"] = rng.choice(PERSONAS)
    counts = plan_answers(attrs, rng)
    n_scores = rng.randint(4, 6)
    attrs["counts"], attrs["n_scores"], attrs["format"] = counts, n_scores, "codes"
    nganh = ", ".join(group["nganh"])
    # "để trống" spelt out as the JSON value: asked in words, no writer of the first runs left the text empty
    text = ('"" (chuỗi rỗng: em để trống phần này)' if attrs["free_text"] == "để trống" else
            f"{attrs['free_text']}; {attrs['free_text_style']}; {attrs['mentions_job']}")
    prompt = PREFIX + (
        "HỌC SINH CẦN TƯỞNG TƯỢNG\n"
        f"Nhóm ngành hợp với em: {group['name']} (gồm các ngành: {nganh}).\n"
        f"Em là {attrs['persona']}; {attrs['level']}; {attrs['clarity']}; {attrs['subjects']}.\n"
        f"Số mã chọn ở mỗi câu: {plan_text(counts)}.\n"
        f"diem: ĐÚNG {n_scores} môn khác nhau.\n"
        f"ke_them: {text}."
    )
    return prompt, attrs

def parse(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        data = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    return data if isinstance(data, dict) else None

def known(q, x):
    # An answer naming one of the question's options: its code (any case) or, in the first runs' files, its label.
    return isinstance(x, str) and (x.strip().lower() in {c.lower() for c in q["codes"]} or x in q["options"])

def follows(answers, attrs):
    # Problems with one student against its plan: {"count": [...], "foreign": [...], "skip": [...], "scores": n}
    out = {"count": [], "foreign": [], "skip": [], "scores": 0}
    for q in QUESTIONS:
        k, a = attrs["counts"][q["id"]], answers.get(q["id"])
        picked = [] if a is None else ([a] if isinstance(a, str) else list(a))
        if k is None:
            if picked:
                out["skip"].append(q["id"])
            continue
        if len(picked) != k:
            out["count"].append(q["id"])
        out["foreign"] += [f"{q['id']}:{x}" for x in picked if not known(q, x)]
    diem = answers.get("diem")
    out["scores"] = len(diem) if isinstance(diem, dict) else 0
    return out

def summary(rows):
    checked = [(json.loads(r["answers"]), json.loads(r["attributes"])) for r in rows if r["answers"]]
    if not checked:
        return "nothing parsed"
    f = [follows(a, t) for a, t in checked]
    n = len(f)
    return (f"{n} parsed of {len(rows)}; wrong number of picks: {sum(len(x['count']) for x in f)} questions in "
            f"{sum(bool(x['count']) for x in f)} students; options from another question or misspelt: "
            f"{sum(len(x['foreign']) for x in f)}; skipped questions answered: {sum(len(x['skip']) for x in f)}; "
            f"scores as asked: {sum(x['scores'] == t['n_scores'] for x, (_, t) in zip(f, checked))}/{n}")

def row(sid, group, family, attrs, text, engine):
    answers = parse(text)
    return {"student_id": sid, "target_group": group, "family": family, "model": MODELS[family][0],
            "attributes": json.dumps({**attrs, "engine": engine}, ensure_ascii=False), "raw": text,
            "answers": json.dumps(answers, ensure_ascii=False) if answers else ""}

def save(rows, family, final=False):
    tag = f"{family}_trial" if TRIAL else family
    df = pd.DataFrame(rows)
    df.to_csv(f"/kaggle/working/student_generated_{tag}.csv", index=False)
    if final and len(df):
        (df.sample(frac=1, random_state=SEED).groupby("target_group").head(3).sort_values("target_group")
           .to_csv(f"/kaggle/working/student_review_sample_{tag}.csv", index=False))
    return df

def previous(family):
    # Students of an earlier run of this family: any CSV attached as an input dataset (whatever Kaggle named it) with
    # this notebook's columns and only this family's students.
    rows = []
    files = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
    for path in files:
        if os.path.basename(path).startswith("student_review_sample"):   # a sample of a run, not more students
            continue
        try:
            df = pd.read_csv(path, dtype=str, keep_default_na=False)
        except Exception:
            continue
        if {"student_id", "family", "answers", "raw"} <= set(df.columns) and len(df) and set(df["family"]) == {family}:
            rows += df.to_dict("records")
            print("resuming from", path, len(df), "students")
    if not rows and REQUIRE_PREVIOUS:
        print("CSV files under /kaggle/input:", files or "none")
        raise SystemExit(f"No earlier {family} file is attached: add the dataset with student_generated_{family}.csv "
                         "(Add Input), then run again. Stopped before loading the model, so no GPU time is used.")
    return list({r["student_id"]: r for r in rows}.values())

# ---- vLLM engine: its own process (a failed load cannot take the notebook down), results appended per chunk ----
WORKER = '''import json, sys, time
from vllm import LLM, SamplingParams

def main():
    job = json.load(open(sys.argv[1], encoding="utf-8"))
    t0 = time.time()
    llm = LLM(model=job["model"], seed=job["seed"], **job["engine"])
    print(f"vLLM: {job['model']} loaded in {time.time() - t0:.0f} s", flush=True)
    params = SamplingParams(temperature=job["temperature"], top_p=job["top_p"], max_tokens=job["max_tokens"],
                            seed=job["seed"])
    tok = llm.get_tokenizer()
    items = job["items"]
    with open(job["out"], "a", encoding="utf-8") as f:
        for b in range(0, len(items), job["chunk"]):
            if time.time() > job["deadline"]:
                print("vLLM: stopped at the time limit", flush=True)
                break
            chunk = items[b:b + job["chunk"]]
            if tok.chat_template:
                outs = llm.chat([[{"role": "user", "content": p}] for _, p in chunk], params, use_tqdm=False)
            else:   # SeaLLM-7B-v2 ships no template: its documented format (vLLM adds the BOS itself)
                prompts = [f"<|im_start|>user\\n{p}</s><|im_start|>assistant\\n" for _, p in chunk]
                outs = llm.generate(prompts, params, use_tqdm=False)
            for (sid, _), o in zip(chunk, outs):
                f.write(json.dumps({"student_id": sid, "text": o.outputs[0].text}, ensure_ascii=False) + "\\n")
            f.flush()
            print(f"vLLM: {b + len(chunk)}/{len(items)} students, {time.time() - t0:.0f} s", flush=True)

# vLLM may start its engine processes with "spawn", which imports this file again: run only as the script
if __name__ == "__main__":
    main()
'''

def run_vllm(family, todo, deadline):
    # {student_id: text} written by vLLM, and the worker's exit code (0: finished or stopped at the time limit).
    work = "/kaggle/working"
    out, job_path, worker = f"{work}/vllm_{family}.jsonl", f"{work}/vllm_job_{family}.json", f"{work}/vllm_worker.py"
    if os.path.exists(out):
        os.remove(out)
    with open(worker, "w", encoding="utf-8") as f:
        f.write(WORKER)
    job = {"model": MODELS[family][0], "engine": {**VLLM, **VLLM_MODEL.get(family, {})}, "seed": SEED,
           "temperature": TEMPERATURE, "top_p": TOP_P, "max_tokens": MAX_NEW_TOKENS, "chunk": CHUNK,
           "deadline": deadline, "out": out, "items": [[j[0], j[2]] for j in todo]}
    with open(job_path, "w", encoding="utf-8") as f:
        json.dump(job, f, ensure_ascii=False)
    proc = subprocess.Popen([sys.executable, worker, job_path], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, env=dict(os.environ))
    tail = []
    for line in proc.stdout:              # the worker's log, shown as it runs
        tail = (tail + [line])[-40:]
        if line.startswith("vLLM:") or "Error" in line or "error" in line[:80]:
            print(line, end="")
    code = proc.wait()
    if code:
        print("".join(tail))
    texts = {}
    if os.path.exists(out):
        for line in open(out, encoding="utf-8"):
            r = json.loads(line)
            texts[r["student_id"]] = r["text"]
    return texts, code

# ---- transformers engine (the first runs'): slow, known to work for every family ----
def load(model_id, how):
    tok = AutoTokenizer.from_pretrained(model_id, token=os.environ.get("HF_TOKEN"))
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = dict(device_map="auto", token=os.environ.get("HF_TOKEN"))
    if how == "fp16":
        kwargs["torch_dtype"] = torch.float16
    elif how == "4bit_fp32":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float32)
        kwargs["attn_implementation"] = "eager"
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    return tok, model

def chat_text(tok, prompt):
    messages = [{"role": "user", "content": prompt}]   # no system role: Gemma has none
    if tok.chat_template:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Fallback when the tokenizer ships no template: SeaLLM-7B-v2's documented format (one BOS, no newline after </s>).
    return f"<s><|im_start|>user\n{prompt}</s><|im_start|>assistant\n"


def run_transformers(family, todo, rows, deadline):
    model_id, how = MODELS[family]
    tok, model = load(model_id, how)
    start = time.time()
    for i, b in enumerate(range(0, len(todo), BATCH_SIZE)):
        if time.time() > deadline:
            print(f"{family}: stopped after {MAX_HOURS} h to stay inside Kaggle's limit; resume in a new session")
            break
        batch = todo[b:b + BATCH_SIZE]
        enc = tok([chat_text(tok, j[2]) for j in batch], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE,
                                 top_p=TOP_P, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        rows += [row(sid, group, family, attrs, text, "transformers")
                 for (sid, group, _, attrs), text in zip(batch, texts)]
        if i % SAVE_EVERY == 0:
            save(rows, family)
            ok = sum(bool(r["answers"]) for r in rows)
            print(f"{family}: {b + len(batch)}/{len(todo)} students, {ok} parsed, {time.time() - start:.0f}s")
    del model
    gc.collect()
    torch.cuda.empty_cache()

def run_family(family):
    rng = random.Random(f"{SEED}-{family}")
    jobs = []
    for group in (GROUPS[:3] if TRIAL else GROUPS):
        for k in range(2 if TRIAL else STUDENTS_PER_GROUP[family]):
            prompt, attrs = build_prompt(group, rng)
            jobs.append((f"{family}-{group['code']}-{k}", group["code"], prompt, attrs))
    rng.shuffle(jobs)                      # groups mixed in every chunk: a stopped run still covers all groups
    rows = previous(family)
    if TRIAL:                              # a trial checks the attached file but writes fresh students to *_trial.csv
        print(f"{family}: trial; the {len(rows)} students of the attached file are left as they are")
        rows = []
    done = {r["student_id"] for r in rows}
    jobs = [j for j in jobs if j[0] not in done]
    print(f"{family}: {len(rows)} students from an earlier run kept, {len(jobs)} to generate")
    deadline = time.time() + MAX_HOURS * 3600
    for engine in ENGINES[family]:
        todo = [j for j in jobs if j[0] not in {r["student_id"] for r in rows}]
        if not todo or time.time() > deadline:
            break
        if engine == "vllm":
            if not VLLM_OK:
                continue
            start = time.time()
            texts, code = run_vllm(family, todo, deadline)
            rows += [row(sid, group, family, attrs, texts[sid], "vllm") for sid, group, _, attrs in todo if sid in texts]
            save(rows, family)
            if len(texts):
                print(f"{family}: vLLM wrote {len(texts)} students in {time.time() - start:.0f} s "
                      f"({(time.time() - start) / len(texts):.2f} s per student, model loading included)")
            if code == 0:
                break
            print(f"{family}: vLLM stopped with exit code {code} (log above); the next engine takes over")
        else:
            run_transformers(family, todo, rows, deadline)
            break
    df = save(rows, family, final=True)
    print(family, "done:", len(df), "students,", int((df["answers"] != "").sum()), "parsed")
    print(summary(rows))
    if TRIAL:
        for r in rows:
            print("\n", r["target_group"], r["attributes"], "\n", r["raw"][:1500])
    return df

results = {f: run_family(f) for f in RUN_FAMILIES}

## After the run

1. Download `student_generated_<family>.csv` into `data/step7/generated/` and `student_review_sample_<family>.csv`
   into `data/step7/review/` of the repository.
2. People read the review samples (spec 3.5: "People review a sample of generated students"): does the student
   believably suit the target group? Is another group a better fit?
3. Then clean the answers (labels mapped back to option codes), audits writer habits (how often each
   group's students tick each option against everyone), and trains the model.